# 06 — GPU-Accelerated Validation + Notion Summary

**Run on H100 GPU node.**

Uses rapids-singlecell for standardized DE (~15 min total vs ~14 hours CPU).

**Important:** For Mouse Brain 1M, creates a 1M-cell canonical subset to avoid OOM
(full 1.15M × 22K genes exceeds 95GB VRAM when dense).

In [1]:
import cupy as cp
print(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
print(f"VRAM: {cp.cuda.runtime.memGetInfo()[1] / 1e9:.1f} GB")

GPU: NVIDIA H100 NVL
VRAM: 99.9 GB


In [2]:
import json, os, time, glob
import pandas as pd
import numpy as np
import scanpy as sc

CONFIG_PATH = "benchmark_config.json"
DATASETS = ["pbmc3k", "lung65k", "mouse_brain_1m"]

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

print(f"Scanpy: {sc.__version__}")
os.makedirs("write", exist_ok=True)

Scanpy: 1.12


/tmp/ipykernel_1245332/4214173363.py:12: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"Scanpy: {sc.__version__}")


## Step 0: Prepare 1M-cell canonical subset for Mouse Brain

The full 1.15M × 22K matrix exceeds H100 VRAM when dense.
rsc and ScaleSC only processed 1M cells anyway, so validation only needs 1M.

In [3]:
mouse_cfg = cfg["datasets"]["mouse_brain_1m"]
full_h5ad = mouse_cfg["canonical_h5ad"]
subset_h5ad = full_h5ad.replace(".h5ad", "_1M.h5ad")

if not os.path.exists(subset_h5ad):
    print(f"Creating 1M subset from {full_h5ad}...")
    adata = sc.read_h5ad(full_h5ad)
    print(f"  Full: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
    adata = adata[:1_000_000, :].copy()
    adata.write_h5ad(subset_h5ad)
    print(f"  Saved: {subset_h5ad} ({adata.n_obs:,} cells)")
    del adata
else:
    print(f"Already exists: {subset_h5ad}")

# Update config to point to 1M subset for validation
cfg["datasets"]["mouse_brain_1m"]["canonical_h5ad"] = subset_h5ad

# Write temp config
TEMP_CONFIG = "benchmark_config_validation.json"
with open(TEMP_CONFIG, "w") as f:
    json.dump(cfg, f, indent=2)
print(f"Wrote temp config: {TEMP_CONFIG}")

Creating 1M subset from write/mouse_1m_canonical_filtered.h5ad...
  Full: 1,153,539 cells x 22,788 genes
  Saved: write/mouse_1m_canonical_filtered_1M.h5ad (1,000,000 cells)
Wrote temp config: benchmark_config_validation.json


## Step 1: Run GPU validation for all datasets

In [4]:
import subprocess, sys

SCRIPT = "validate_cross_pipeline_gpu.py"

validation_results = {}
for ds in DATASETS:
    print(f"\n{'='*72}")
    print(f"GPU Validation: {ds}")
    print(f"{'='*72}")
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, SCRIPT,
         "--dataset", ds, "--config", TEMP_CONFIG, "--gpu"],
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0
    validation_results[ds] = {"returncode": result.returncode, "elapsed": elapsed}
    
    if result.returncode == 0:
        print(result.stdout)
        print(f"Completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
    else:
        print(f"FAILED (return code {result.returncode})")
        print("STDOUT:", result.stdout[-2000:] if result.stdout else "(empty)")
        print("STDERR:", result.stderr[-2000:] if result.stderr else "(empty)")

print(f"\n{'='*72}")
print("Validation timing summary:")
total_time = 0
for ds, r in validation_results.items():
    status = "OK" if r["returncode"] == 0 else "FAILED"
    print(f"  {ds:20s}: {r['elapsed']:8.1f}s ({r['elapsed']/60:.1f} min) [{status}]")
    total_time += r["elapsed"]
print(f"  {'TOTAL':20s}: {total_time:8.1f}s ({total_time/60:.1f} min)")


GPU Validation: pbmc3k
GPU mode: RMM initialized, using rapids-singlecell for standardized DE
Validation — PBMC3k
Pipelines detected:
  scanpy_cpu   write/pbmc3k_scanpy_cpu_harmonized_clusters.csv
  seurat_cpu   write/pbmc3k_seurat_cpu_harmonized_clusters.csv
  scalesc_gpu  write/pbmc3k_scalesc_gpu_harmonized_clusters.csv
  rsc_gpu_0141 write/pbmc3k_rsc_gpu_0141_harmonized_clusters.csv
  rsc_gpu_015  write/pbmc3k_rsc_gpu_015_harmonized_clusters.csv

  Standardized DE for scanpy_cpu (GPU)... 6.6s
  Standardized DE for seurat_cpu (GPU)... 1.4s
  Standardized DE for scalesc_gpu (GPU)... 1.2s
  Standardized DE for rsc_gpu_0141 (GPU)... 1.2s
  Standardized DE for rsc_gpu_015 (GPU)... 1.2s
Summary:
dataset                  comparison  common_cells  clusters_a  clusters_b      ARI      NMI  mean_dice  mean_native_deg_jaccard  mean_native_deg_spearman  mean_standardized_deg_jaccard  mean_standardized_deg_spearman  mean_module_profile_rho
 PBMC3k    scanpy_cpu vs seurat_cpu          2638      

## Step 2: Load all results

In [5]:
# Reload original config for display
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

all_summaries = []
all_details = {}

for ds in DATASETS:
    prefix = cfg["datasets"][ds]["pipeline_prefix"]
    val_dir = f"write/validation_{prefix}_harmonized"
    summary_csv = f"{val_dir}/{prefix}_validation_summary.csv"
    detail_json = f"{val_dir}/{prefix}_validation_details.json"
    
    if os.path.exists(summary_csv):
        df = pd.read_csv(summary_csv)
        all_summaries.append(df)
        print(f"Loaded {ds}: {len(df)} comparisons")
    else:
        print(f"MISSING: {summary_csv}")
    
    if os.path.exists(detail_json):
        with open(detail_json) as f:
            all_details[ds] = json.load(f)

if all_summaries:
    full_summary = pd.concat(all_summaries, ignore_index=True)
    print(f"\nTotal comparisons across all datasets: {len(full_summary)}")

Loaded pbmc3k: 10 comparisons
Loaded lung65k: 10 comparisons
Loaded mouse_brain_1m: 3 comparisons

Total comparisons across all datasets: 23


## Step 3: Per-dataset comparison tables

In [6]:
for ds in DATASETS:
    if ds not in all_details:
        print(f"\nSkipping {ds} (no details)")
        continue
    
    ds_name = cfg["datasets"][ds]["name"]
    print(f"\n{'='*72}")
    print(f"{ds_name}")
    print(f"{'='*72}")
    
    rows = []
    for pair_key, metrics in all_details[ds].items():
        parts = pair_key.split("__vs__")
        if len(parts) != 2:
            continue
        rows.append({
            "Comparison": f"{parts[0]} vs {parts[1]}",
            "Clusters": f"{metrics['n_clusters_a']} vs {metrics['n_clusters_b']}",
            "ARI": round(metrics["ARI"], 3),
            "NMI": round(metrics["NMI"], 3),
            "Dice": round(metrics.get("mean_dice", 0), 3),
            "Std DEG Jac": round(metrics["mean_standardized_deg_jaccard"], 3) if metrics.get("mean_standardized_deg_jaccard") else "-",
            "Std DEG \u03C1": round(metrics["mean_standardized_deg_spearman"], 3) if metrics.get("mean_standardized_deg_spearman") else "-",
            "Nat DEG \u03C1": round(metrics["mean_native_deg_spearman"], 3) if metrics.get("mean_native_deg_spearman") else "-",
            "Module \u03C1": round(metrics["mean_module_profile_rho"], 3) if metrics.get("mean_module_profile_rho") else "-",
        })
    print(pd.DataFrame(rows).to_string(index=False))


PBMC3k
                 Comparison Clusters   ARI   NMI  Dice  Std DEG Jac  Std DEG ρ Nat DEG ρ  Module ρ
   scanpy_cpu vs seurat_cpu  9 vs 10 0.655 0.762 0.790        0.736      0.830     0.877     0.980
  scanpy_cpu vs scalesc_gpu   9 vs 9 0.761 0.824 0.835        0.778      0.851         -     0.984
 scanpy_cpu vs rsc_gpu_0141   9 vs 9 0.923 0.935 0.980        0.943      0.983     0.983     1.000
  scanpy_cpu vs rsc_gpu_015   9 vs 9 0.923 0.935 0.980        0.943      0.983     0.983     1.000
  seurat_cpu vs scalesc_gpu  10 vs 9 0.662 0.763 0.857        0.830      0.848         -     0.992
 seurat_cpu vs rsc_gpu_0141  10 vs 9 0.639 0.750 0.782        0.725      0.821     0.874     0.980
  seurat_cpu vs rsc_gpu_015  10 vs 9 0.639 0.750 0.782        0.725      0.821     0.874     0.980
scalesc_gpu vs rsc_gpu_0141   9 vs 9 0.748 0.816 0.829        0.773      0.849         -     0.984
 scalesc_gpu vs rsc_gpu_015   9 vs 9 0.748 0.816 0.829        0.773      0.849         -     0.984
rs

## Step 4: Runtime comparison

In [7]:
print("Runtime comparison")
print("=" * 72)

for ds in DATASETS:
    prefix = cfg["datasets"][ds]["pipeline_prefix"]
    ds_name = cfg["datasets"][ds]["name"]
    spec_files = sorted(glob.glob(f"write/{prefix}_*_spec.json"))
    
    if not spec_files:
        continue
    
    print(f"\n--- {ds_name} ---")
    for sf in spec_files:
        with open(sf) as f:
            spec = json.load(f)
        pipeline = spec.get("pipeline", os.path.basename(sf))
        timings = spec.get("timings_seconds", {})
        total = sum(timings.values()) if timings else 0
        n_cells = spec.get("results", {}).get("n_cells", "?")
        n_clusters = spec.get("results", {}).get("n_clusters", "?")
        print(f"  {pipeline:30s} | {str(n_cells):>10s} cells | {str(n_clusters):>3s} clusters | {total:8.1f}s ({total/60:.1f} min)")

Runtime comparison

--- PBMC3k ---
  rsc_gpu_harmonized             |       2638 cells |   9 clusters |      0.0s (0.0 min)
  rsc_gpu_harmonized             |       2638 cells |   9 clusters |      0.0s (0.0 min)
  rsc_gpu_harmonized             |       2638 cells |   9 clusters |      0.0s (0.0 min)
  scalesc_gpu_harmonized         |       2638 cells |   9 clusters |      3.4s (0.1 min)
  scanpy_cpu_harmonized          |       2638 cells |   9 clusters |      0.0s (0.0 min)
  seurat_cpu_harmonized          |       2638 cells |  10 clusters |      0.0s (0.0 min)

--- Human Lung Cell 65K ---
  rsc_gpu_harmonized             |      65462 cells |  37 clusters |      0.0s (0.0 min)
  rsc_gpu_harmonized             |      65462 cells |  38 clusters |      0.0s (0.0 min)
  rsc_gpu_harmonized             |      65462 cells |  37 clusters |      0.0s (0.0 min)
  scalesc_gpu_harmonized         |      65462 cells |  38 clusters |      5.4s (0.1 min)
  scanpy_cpu_harmonized          |      65462 

## Step 5: Mouse Brain GPU speedup (step-by-step)

In [8]:
prefix = cfg["datasets"]["mouse_brain_1m"]["pipeline_prefix"]
scanpy_spec_path = f"write/{prefix}_scanpy_cpu_harmonized_spec.json"
rsc_spec_path = f"write/{prefix}_rsc_gpu_harmonized_spec.json"

if os.path.exists(scanpy_spec_path) and os.path.exists(rsc_spec_path):
    with open(scanpy_spec_path) as f:
        s_spec = json.load(f)
    with open(rsc_spec_path) as f:
        r_spec = json.load(f)
    
    s_t = s_spec.get("timings_seconds", {})
    r_t = r_spec.get("timings_seconds", {})
    
    print("Mouse Brain: Scanpy CPU vs rsc GPU")
    print(f"  Scanpy: {s_spec.get('results',{}).get('n_cells','?'):,} cells")
    print(f"  rsc:    {r_spec.get('results',{}).get('n_cells','?'):,} cells")
    print()
    
    steps = ["normalize_log1p", "hvg", "pca", "neighbors", "leiden", "umap", "de"]
    rows = []
    for step in steps:
        s_time = s_t.get(step)
        r_time = r_t.get(step)
        if s_time and r_time and r_time > 0:
            rows.append({
                "Step": step,
                "Scanpy CPU (s)": round(s_time, 1),
                "rsc GPU (s)": round(r_time, 1),
                "Speedup": f"{s_time/r_time:.0f}x",
            })
    
    s_total = sum(s_t.get(s, 0) for s in steps)
    r_total = sum(r_t.get(s, 0) for s in steps)
    if r_total > 0:
        rows.append({
            "Step": "TOTAL (excl I/O)",
            "Scanpy CPU (s)": round(s_total, 1),
            "rsc GPU (s)": round(r_total, 1),
            "Speedup": f"{s_total/r_total:.0f}x",
        })
    
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("Missing spec files for mouse brain speedup comparison")

Mouse Brain: Scanpy CPU vs rsc GPU
  Scanpy: 1,153,539 cells
  rsc:    1,000,000 cells

            Step  Scanpy CPU (s)  rsc GPU (s) Speedup
 normalize_log1p             8.5          4.7      2x
             hvg            58.8          8.1      7x
             pca           129.5          3.7     35x
       neighbors           444.6         13.0     34x
          leiden          1268.0          3.2    398x
            umap          1010.7          3.8    265x
              de         16517.9         38.1    434x
TOTAL (excl I/O)         19438.1         74.6    261x


## Step 6: Notion-ready markdown (copy-paste)

In [9]:
def df_to_md(df, title=""):
    lines = []
    if title:
        lines.append(f"### {title}\n")
    cols = df.columns.tolist()
    lines.append("| " + " | ".join(str(c) for c in cols) + " |")
    lines.append("| " + " | ".join("---" for _ in cols) + " |")
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(v) for v in row.values) + " |")
    return "\n".join(lines)


print("\n" + "=" * 72)
print("COPY-PASTE FOR NOTION")
print("=" * 72)

for ds in DATASETS:
    if ds not in all_details:
        continue
    ds_name = cfg["datasets"][ds]["name"]
    
    rows = []
    for pair_key, metrics in all_details[ds].items():
        parts = pair_key.split("__vs__")
        if len(parts) != 2:
            continue
        rows.append({
            "Comparison": f"{parts[0]} vs {parts[1]}",
            "Clusters": f"{metrics['n_clusters_a']} vs {metrics['n_clusters_b']}",
            "ARI": round(metrics["ARI"], 3),
            "NMI": round(metrics["NMI"], 3),
            "Dice": round(metrics.get("mean_dice", 0), 3),
            "DEG Jac": round(metrics.get("mean_standardized_deg_jaccard", 0), 3) if metrics.get("mean_standardized_deg_jaccard") else "-",
            "DEG \u03C1": round(metrics.get("mean_standardized_deg_spearman", 0), 3) if metrics.get("mean_standardized_deg_spearman") else "-",
            "Module \u03C1": round(metrics.get("mean_module_profile_rho", 0), 3) if metrics.get("mean_module_profile_rho") else "-",
        })
    
    print(f"\n{df_to_md(pd.DataFrame(rows), title=ds_name)}")
    print()


COPY-PASTE FOR NOTION

### PBMC3k

| Comparison | Clusters | ARI | NMI | Dice | DEG Jac | DEG ρ | Module ρ |
| --- | --- | --- | --- | --- | --- | --- | --- |
| scanpy_cpu vs seurat_cpu | 9 vs 10 | 0.655 | 0.762 | 0.79 | 0.736 | 0.83 | 0.98 |
| scanpy_cpu vs scalesc_gpu | 9 vs 9 | 0.761 | 0.824 | 0.835 | 0.778 | 0.851 | 0.984 |
| scanpy_cpu vs rsc_gpu_0141 | 9 vs 9 | 0.923 | 0.935 | 0.98 | 0.943 | 0.983 | 1.0 |
| scanpy_cpu vs rsc_gpu_015 | 9 vs 9 | 0.923 | 0.935 | 0.98 | 0.943 | 0.983 | 1.0 |
| seurat_cpu vs scalesc_gpu | 10 vs 9 | 0.662 | 0.763 | 0.857 | 0.83 | 0.848 | 0.992 |
| seurat_cpu vs rsc_gpu_0141 | 10 vs 9 | 0.639 | 0.75 | 0.782 | 0.725 | 0.821 | 0.98 |
| seurat_cpu vs rsc_gpu_015 | 10 vs 9 | 0.639 | 0.75 | 0.782 | 0.725 | 0.821 | 0.98 |
| scalesc_gpu vs rsc_gpu_0141 | 9 vs 9 | 0.748 | 0.816 | 0.829 | 0.773 | 0.849 | 0.984 |
| scalesc_gpu vs rsc_gpu_015 | 9 vs 9 | 0.748 | 0.816 | 0.829 | 0.773 | 0.849 | 0.984 |
| rsc_gpu_0141 vs rsc_gpu_015 | 9 vs 9 | 1.0 | 1.0 | 1.0 | 1.0 

## Step 7: Save CSVs + cleanup

In [10]:
# Save consolidated summary
if all_summaries:
    full_summary.to_csv("write/all_datasets_validation_summary.csv", index=False)
    print("Saved: write/all_datasets_validation_summary.csv")

# Clean up temp config
if os.path.exists(TEMP_CONFIG):
    os.remove(TEMP_CONFIG)
    print(f"Cleaned up: {TEMP_CONFIG}")

print("\nAll validation outputs:")
for d in sorted(glob.glob("write/validation_*_harmonized/")):
    print(f"  {d}")
    for f in sorted(glob.glob(d + "*_summary.csv") + glob.glob(d + "*_details.json")):
        print(f"    {os.path.basename(f)}")

print("\nDone! Copy the Notion markdown from Step 6 above.")

Saved: write/all_datasets_validation_summary.csv
Cleaned up: benchmark_config_validation.json

All validation outputs:
  write/validation_lung_65k_harmonized/
    lung_65k_validation_details.json
    lung_65k_validation_summary.csv
  write/validation_mouse_1m_harmonized/
    mouse_1m_validation_details.json
    mouse_1m_validation_summary.csv
  write/validation_pbmc3k_harmonized/
    pbmc3k_validation_details.json
    pbmc3k_validation_summary.csv

Done! Copy the Notion markdown from Step 6 above.
